In [12]:
import torch
import torchvision
from torch.utils.data import DataLoader
import torch.optim 
import torch.nn as nn
import torchvision.transforms as tranforms

In [13]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='mps')

In [14]:
train=torchvision.datasets.MNIST(root="./data",train=True,transform=tranforms.ToTensor())
test=torchvision.datasets.MNIST(root="./data",train=False,transform=tranforms.ToTensor())

In [15]:
train_loader=DataLoader(train,batch_size=64,shuffle=True)

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.net=nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3),   # (1, 28, 28) → (32, 26, 26)
            nn.ReLU(),
            nn.MaxPool2d(2),                    # → (32, 13, 13)
            nn.Conv2d(32, 64, kernel_size=3),   # → (64, 11, 11)
            nn.ReLU(),
            nn.MaxPool2d(2),                    # → (64, 5, 5)
            nn.Flatten(),                       # → 64*5*5 = 1600
            nn.Linear(1600, 128),
            nn.ReLU(),
            nn.Linear(128, 10)


        )
    def forward(self, x):
        return self.net(x)

In [17]:
Model=CNN().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(Model.parameters(),lr=0.001)

In [19]:
#training looop

for epoch in range(5):
    for X,y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        outputs=Model(X)

        loss=criterion(outputs,y)
        loss.backward()

        optimizer.step()


    print(f"Epoch {epoch+1}, Loss: {loss.item()}")
print("all done")

Epoch 1, Loss: 0.0037942780181765556
Epoch 2, Loss: 0.041771337389945984
Epoch 3, Loss: 0.0223014447838068
Epoch 4, Loss: 0.01163283921778202
Epoch 5, Loss: 0.006643841974437237
all done


In [21]:
#test loop
test_load=DataLoader(test,batch_size=64,shuffle=True)


correct=0
total=0
for X,y in test_load:
     X = X.to(device)
     y = y.to(device)

     output=Model(X)
     predictions = output.argmax(dim=1)
     correct += (predictions == y).sum().item()
     total += y.size(0)


print(f"Test Accuracy: {100 * correct / total:.2f}%")
print(total)

Test Accuracy: 99.00%
10000
